## Download the data

Every modeling project starts with the same first step: get the data.


In [ ]:
# Make the project root (the parent of 'notebooks/') importable
import sys, pathlib
cwd = pathlib.Path.cwd()
if cwd.name == "notebooks":
    proj_root = cwd.parent
else:
    proj_root = cwd  # if you launched from project root
if str(proj_root) not in sys.path:
    sys.path.insert(0, str(proj_root))

In [ ]:
from pathlib import Path

In [ ]:
from scripts.ingest.download_pumas_geojson import PumaGeoJSONDownloader
from scripts.ingest.download_nyc_311_noise import NYC311NoiseDownloader
from scripts.ingest.download_nta_geojson import NtaGeoJSONDownloader
from scripts.lookups.build_puma_nta_lookup import PumaNtaLookupBuilder
from scripts.aggregate.build_noise_counts_with_lookup import PumaNoiseCountsWithLookup

### Download PUMA Geojson Data

For geography, we use two spatial layers: PUMAs and NTAs.

PUMA (Public Use Microdata Area) is a Census-defined geography created specifically for statistical analysis — not for mail delivery or administrative use.

Key properties:

  - Population between roughly 100,000–200,000
  - Stable boundaries over time

We choose PUMAs over ZIP codes because ZIP codes aren’t designed for analysis. They change, split neighborhoods awkwardly, and have irregular shapes. PUMAs are more stable and statistically meaningful.

In [ ]:
puma_geojson_dl = PumaGeoJSONDownloader(out_dir = Path("../data/raw/nyc/geographies"))

In [ ]:
_ =puma_geojson_dl.download()

### Download NTA GeoJson Data

Next, we use Neighborhood Tabulation Areas (NTAs).

NTAs were created by NYC’s Department of City Planning to analyze populations at a neighborhood level.

Key properties:
  - Designed with a minimum population of about 15,000
  - More recognizable and intuitive than numeric PUMA codes

We use NTAs to group and label PUMAs, helping us uncover complaint relationships while keeping the geography interpretable.

In [ ]:
dl = NtaGeoJSONDownloader(
    out_dir=Path("../data/raw/nyc/geographies")
)

In [ ]:
_ =  dl.download(filename="nyc_ntas_2020.geojson")

### Download Summer NYC Noise data

Finally, we extract 311 noise complaints and focus specifically on summertime — June through August — from 2021 to 2024.

2021–2024 → training data

Summer 2025 → testing and validation

In [ ]:
nyc_311_noise_dl =  NYC311NoiseDownloader(
                                          query_file=Path("../queries/nyc_311_noise.soql"),
                                          out_dir=Path("../data/raw/nyc/311_noise")  
                                         )

In [ ]:
nyc_311_noise_dl.run(start_date="2021-07-01", end_date="2021-08-31")

In [ ]:
nyc_311_noise_dl.run(start_date="2022-06-01", end_date="2022-08-31")

In [ ]:
nyc_311_noise_dl.run(start_date="2023-06-01", end_date="2023-08-31")

In [ ]:
nyc_311_noise_dl.run(start_date="2024-06-01", end_date="2024-08-31")

In [ ]:
nyc_311_noise_dl.run(start_date="2025-06-01", end_date="2025-08-31")

### Build PUMA NTA Lookup File

In [ ]:
builder = PumaNtaLookupBuilder(
    puma_geojson="../data/raw/nyc/geographies/nyc_pumas_2020.geojson",
    nta_geojson= "../data/raw/nyc/geographies/nyc_ntas_2020.geojson",
    out_dir="../data/processed/lookup",
    out_stem="puma_nta_lookup",
    formats=["parquet"],
)

In [ ]:
_ = builder.run()

### Aggregate Counts using PUMA NTA Lookup Data

In [ ]:
runner = PumaNoiseCountsWithLookup(
    puma_geojson="../data/raw/nyc/geographies/nyc_pumas_2020.geojson",
    out_dir="../data/processed/features",
    out_stem_puma="puma_noise_counts",
    out_stem_geo="nta_noise_counts",
    predicate="within",
    time_bucket="day",
    tod_scheme="two",
    drop_unmatched=True,
    formats=["parquet"],
    lookup_csv="../data/processed/lookup/puma_nta_lookup.parquet",
    lookup_puma_col="puma",
    lookup_geo_col="nta",
    lookup_geo_name_col="nta_name",
    lookup_weight_col="area_share_of_puma",
    geo_level="puma",
)

In [ ]:
_ = runner.run("../data/raw/nyc/311_noise/by_month")